In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
import random

# Seed for reproducibility
np.random.seed(42)
random.seed(42)

# Output path
RAW_PATH = r"D:\Projects\End-to-end projects\17. SKU Proliferation & Rationalization\Data\Raw"
os.makedirs(RAW_PATH, exist_ok=True)

print("Path ready:", RAW_PATH)

Path ready: D:\Projects\End-to-end projects\17. SKU Proliferation & Rationalization\Data\Raw


In [2]:
# --- Configuration ---
N_SKUS = 280

categories = {
    "Haircare":   ["Shampoo", "Conditioner", "Hair Oil", "Hair Mask"],
    "Skincare":   ["Face Wash", "Serum", "Moisturizer", "Sunscreen"],
    "Nutrition":  ["Protein Powder", "Vitamins", "Supplements"],
    "Body Care":  ["Body Lotion", "Body Wash", "Scrub"],
    "Oral Care":  ["Toothpaste", "Mouthwash", "Tooth Serum"],
    "Baby Care":  ["Baby Wash", "Baby Lotion", "Baby Oil"]
}

pack_sizes = ["30ml", "50ml", "100ml", "150ml", "200ml",
              "250g", "500g", "1kg", "30 tablets", "60 capsules"]

# Build SKU list
rows = []
sku_id = 1

for category, subcategories in categories.items():
    # Distribute 280 SKUs proportionally across 6 categories
    n = 47 if category != "Baby Care" else 45
    for i in range(n):
        subcat = random.choice(subcategories)
        mrp    = round(random.uniform(199, 1499), 0)
        cost   = round(mrp * random.uniform(0.28, 0.48), 0)  # 28–48% of MRP

        # Launch dates: most SKUs launched in 2022–2023, some older, some newer
        days_ago   = random.randint(30, 1200)
        launch_date = (datetime(2024, 1, 1) - timedelta(days=days_ago)).date()

        rows.append({
            "sku_id":       f"SKU{str(sku_id).zfill(4)}",
            "sku_name":     f"{category} {subcat} Variant {sku_id}",
            "category":     category,
            "subcategory":  subcat,
            "launch_date":  launch_date,
            "mrp":          mrp,
            "cost_price":   cost,
            "is_active":    random.choices([True, False], weights=[92, 8])[0],
            "pack_size":    random.choice(pack_sizes)
        })
        sku_id += 1

sku_master = pd.DataFrame(rows)

# Save
sku_master.to_csv(os.path.join(RAW_PATH, "sku_master.csv"), index=False)
print(f"sku_master: {sku_master.shape[0]} rows saved")
sku_master.head(3)

sku_master: 280 rows saved


,sku_id,sku_name,category,subcategory,launch_date,mrp,cost_price,is_active,pack_size
0,SKU0001,Haircare Shampoo Variant 1,Haircare,Shampoo,2022-09-01,232.0,78.0,True,50ml
1,SKU0002,Haircare Shampoo Variant 2,Haircare,Shampoo,2023-05-25,967.0,277.0,True,30 tablets
2,SKU0003,Haircare Shampoo Variant 3,Haircare,Shampoo,2020-11-11,929.0,393.0,True,1kg


In [3]:
# --- Configuration ---
N_ORDERS     = 85000
START_DATE   = datetime(2024, 1, 1)
END_DATE     = datetime(2025, 12, 31)
DATE_RANGE   = (END_DATE - START_DATE).days

channels     = ["D2C Website", "Amazon", "Nykaa", "Offline"]
channel_wts  = [0.35, 0.30, 0.25, 0.10]

active_skus  = sku_master[sku_master["is_active"] == True]["sku_id"].tolist()

# SKU popularity — 80/20 rule baked in
#   Top 20% SKUs get 75% of order weight
n_active     = len(active_skus)
top_n        = int(n_active * 0.20)
tail_n       = n_active - top_n

top_skus     = active_skus[:top_n]
tail_skus    = active_skus[top_n:]

top_weights  = [random.uniform(0.8, 1.5) for _ in top_skus]
tail_weights = [random.uniform(0.01, 0.15) for _ in tail_skus]

all_skus    = top_skus + tail_skus
all_weights = top_weights + tail_weights
total_w     = sum(all_weights)
all_weights = [w / total_w for w in all_weights]  # Normalize

# Build transactions
rows = []
for i in range(N_ORDERS):
    order_date  = START_DATE + timedelta(days=random.randint(0, DATE_RANGE))
    sku         = np.random.choice(all_skus, p=all_weights)
    sku_row     = sku_master[sku_master["sku_id"] == sku].iloc[0]

    # Add seasonal spike: Q4 (Oct–Dec) gets 30% more orders — realistic for D2C
    if order_date.month in [10, 11, 12] and random.random() < 0.30:
        pass  # Already included in base N

    discount    = round(random.choices(
                    [0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30],
                    weights=[20, 15, 20, 20, 12, 8, 5])[0], 2)
    sell_price  = round(sku_row["mrp"] * (1 - discount), 2)
    units       = random.choices([1, 2, 3, 4], weights=[60, 25, 10, 5])[0]
    returned    = random.choices([True, False], weights=[8, 92])[0]
    customer_id = f"CUST{str(random.randint(1, 38000)).zfill(6)}"

    rows.append({
        "order_id":      f"ORD{str(i+1).zfill(7)}",
        "order_date":    order_date.date(),
        "customer_id":   customer_id,
        "sku_id":        sku,
        "channel":       random.choices(channels, weights=channel_wts)[0],
        "units_sold":    units,
        "selling_price": sell_price,
        "discount_pct":  discount,
        "return_flag":   returned
    })

sales = pd.DataFrame(rows)
sales["order_date"] = pd.to_datetime(sales["order_date"])

# Save
sales.to_csv(os.path.join(RAW_PATH, "sales_transactions.csv"), index=False)
print(f"sales_transactions: {sales.shape[0]} rows saved")
sales.head(3)

sales_transactions: 85000 rows saved


,order_id,order_date,customer_id,sku_id,channel,units_sold,selling_price,discount_pct,return_flag
0,ORD0000001,2024-03-21,CUST031397,SKU0030,D2C Website,1,276.45,0.05,False
1,ORD0000002,2025-07-05,CUST007159,SKU0238,Amazon,1,987.05,0.05,False
2,ORD0000003,2025-08-23,CUST014790,SKU0054,Amazon,1,1356.60,0.05,False


In [4]:
rows = []

for _, sku in sku_master.iterrows():
    mrp = sku["mrp"]

    # Storage cost scales loosely with MRP (larger/premium products cost more to store)
    storage = round(random.uniform(80, 600), 0)

    # MOQ — higher for mass-market, lower for premium niche
    moq = random.choices([50, 100, 200, 500, 1000], weights=[10, 30, 35, 20, 5])[0]

    # Handling cost per unit
    handling = round(random.uniform(8, 45), 2)

    # Packaging complexity: 1 = simple pouch, 5 = gift box with inserts
    pkg_score = random.choices([1, 2, 3, 4, 5], weights=[15, 30, 30, 18, 7])[0]

    # Supplier lead time
    lead_time = random.choices([7, 14, 21, 30, 45], weights=[10, 35, 30, 18, 7])[0]

    rows.append({
        "sku_id":                    sku["sku_id"],
        "monthly_storage_cost":      storage,
        "min_order_quantity":        moq,
        "avg_handling_cost_per_unit": handling,
        "packaging_complexity_score": pkg_score,
        "supplier_lead_time_days":   lead_time
    })

sku_ops = pd.DataFrame(rows)

sku_ops.to_csv(os.path.join(RAW_PATH, "sku_ops_cost.csv"), index=False)
print(f"sku_ops_cost: {sku_ops.shape[0]} rows saved")
sku_ops.head(3)

sku_ops_cost: 280 rows saved


,sku_id,monthly_storage_cost,min_order_quantity,avg_handling_cost_per_unit,packaging_complexity_score,supplier_lead_time_days
0,SKU0001,510.0,100,9.79,2,14
1,SKU0002,336.0,100,44.44,4,14
2,SKU0003,335.0,100,14.19,1,21


In [5]:
# Derive first purchase per customer from sales data
sales_sorted = sales.sort_values("order_date")

# First purchase per customer
first_purchase = (
    sales_sorted
    .groupby("customer_id")
    .first()
    .reset_index()[["customer_id", "sku_id", "order_date"]]
    .rename(columns={"sku_id": "first_sku_id", "order_date": "first_order_date"})
)

# Second purchase per customer (if exists)
second_purchase = (
    sales_sorted
    .groupby("customer_id")
    .nth(1)
    .reset_index()[["customer_id", "sku_id", "order_date"]]
    .rename(columns={"sku_id": "second_sku_id", "order_date": "second_order_date"})
)

# Total lifetime orders per customer
order_counts = (
    sales
    .groupby("customer_id")
    .agg(total_orders_lifetime=("order_id", "count"))
    .reset_index()
)

# LTV in first 90 days
sales["revenue"] = sales["units_sold"] * sales["selling_price"]

def ltv_90d(group):
    group = group.sort_values("order_date")
    start = group["order_date"].min()
    mask  = group["order_date"] <= start + timedelta(days=90)
    return group.loc[mask, "revenue"].sum()

ltv = (
    sales
    .groupby("customer_id")
    .apply(ltv_90d)
    .reset_index()
    .rename(columns={0: "ltv_90d"})
)

# Merge everything
cfp = (
    first_purchase
    .merge(second_purchase, on="customer_id", how="left")
    .merge(order_counts,    on="customer_id", how="left")
    .merge(ltv,             on="customer_id", how="left")
)

cfp["ltv_90d"] = cfp["ltv_90d"].round(2)

cfp.to_csv(os.path.join(RAW_PATH, "customer_first_purchase.csv"), index=False)
print(f"customer_first_purchase: {cfp.shape[0]} rows saved")
cfp.head(3)

C:\Users\mohsi\AppData\Local\Temp\ipykernel_9868\2875217733.py:42: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(ltv_90d)


customer_first_purchase: 33881 rows saved


,customer_id,first_sku_id,first_order_date,second_sku_id,second_order_date,total_orders_lifetime,ltv_90d
0,CUST000001,SKU0002,2024-08-30,SKU0017,2024-12-07,2,1740.6
1,CUST000002,SKU0270,2024-03-17,SKU0025,2024-09-25,6,989.1
2,CUST000004,SKU0271,2024-10-06,NaN,NaT,1,751.0


In [6]:
print("=== DATASET SUMMARY ===\n")
print(f"SKU Master       : {sku_master.shape[0]} SKUs | {sku_master['category'].nunique()} categories")
print(f"Sales Transactions: {sales.shape[0]} orders | {sales['customer_id'].nunique()} unique customers")
print(f"SKU Ops Cost     : {sku_ops.shape[0]} rows")
print(f"Customer First Purchase: {cfp.shape[0]} customers\n")

print("--- Date Range ---")
print(f"Earliest order: {sales['order_date'].min()}")
print(f"Latest order  : {sales['order_date'].max()}\n")

print("--- Revenue Sanity Check ---")
total_rev = (sales["units_sold"] * sales["selling_price"]).sum()
print(f"Total simulated revenue: ₹{total_rev:,.0f}")

print("\n--- 80/20 Check ---")
sku_rev = (
    sales
    .assign(revenue = sales["units_sold"] * sales["selling_price"])
    .groupby("sku_id")["revenue"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
sku_rev["cumulative_pct"] = sku_rev["revenue"].cumsum() / sku_rev["revenue"].sum()
top_skus_80 = (sku_rev["cumulative_pct"] <= 0.80).sum()
print(f"SKUs driving 80% of revenue: {top_skus_80} out of {len(sku_rev)} active SKUs")
print("\nAll files saved to Raw folder. ✓")

=== DATASET SUMMARY ===

SKU Master       : 280 SKUs | 6 categories
Sales Transactions: 85000 orders | 33881 unique customers
SKU Ops Cost     : 280 rows
Customer First Purchase: 33881 customers

--- Date Range ---
Earliest order: 2024-01-01 00:00:00
Latest order  : 2025-12-31 00:00:00

--- Revenue Sanity Check ---
Total simulated revenue: ₹99,825,434

--- 80/20 Check ---
SKUs driving 80% of revenue: 60 out of 258 active SKUs

All files saved to Raw folder. ✓
